# Google Earth Engine: Downloader ERA5-Land Hourly Multi-Variable (NetCDF)
Notebook ini digunakan untuk mengunduh data reanalisis **ERA5-Land Hourly** (`ECMWF/ERA5_LAND/HOURLY`) dari Google Earth Engine secara presisi untuk 6 variabel atmosfer utama.

### Variabel & Konversi Satuan:
1. **`precipitation`**: Curah hujan (m -> **mm**, `x1000`)
2. **`temperature_2m`**: Suhu udara 2m (K -> **°C**, `-273.15`)
3. **`dewpoint_temperature_2m`**: Suhu titik embun 2m (K -> **°C**, `-273.15`)
4. **`u_wind_10m`**: Zonal wind / angin U 10m (**m/s**)
5. **`v_wind_10m`**: Meridional wind / angin V 10m (**m/s**)
6. **`surface_pressure`**: Tekanan permukaan (Pa -> **hPa**, `/100`)

In [ ]:
!pip install earthengine-api rioxarray geopandas xarray requests

In [ ]:
import os
import json
import time
import requests
import zipfile
import io
import ee
import rioxarray as rxr
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
from google.oauth2.service_account import Credentials
from kaggle_secrets import UserSecretsClient

# Autentikasi GEE via Kaggle Secrets (GEE_KEY)
try:
    user_secrets = UserSecretsClient()
    service_account_info = json.loads(user_secrets.get_secret("GEE_KEY"))
    SCOPES = ['https://www.googleapis.com/auth/earthengine']
    credentials = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
    ee.Initialize(credentials=credentials, project='staklimjerukagung')
    print("✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)")
except Exception as e:
    print(f"⚠️ Gagal Inisialisasi via Secrets, mencoba auth manual: {e}")
    ee.Authenticate()
    ee.Initialize(project='staklimjerukagung')

In [ ]:
# Parameter Wilayah (Bounding Box Kebumen) & Direktori Output
minx, miny, maxx, maxy = 109.3, -7.9, 110.0, -7.4
geometry = ee.Geometry.BBox(minx, miny, maxx, maxy)

folder_induk = Path("data/era5_land")
folder_induk.mkdir(parents=True, exist_ok=True)

tahun_awal = 2005
tahun_akhir = 2026

print(f"Batas Wilayah (Bounding Box Kebumen): Min Lon: {minx}, Min Lat: {miny}, Max Lon: {maxx}, Max Lat: {maxy}")
print(f"Folder Output: {folder_induk.absolute()}")

In [ ]:
def process_era5_image(img):
    # 1. Total Precipitation: m -> mm (* 1000)
    tp = img.select('total_precipitation_hourly').multiply(1000).rename('precipitation')
    # 2. Temperature 2m: K -> °C (- 273.15)
    t2m = img.select('temperature_2m').subtract(273.15).rename('temperature_2m')
    # 3. Dewpoint Temperature 2m: K -> °C (- 273.15)
    d2m = img.select('dewpoint_temperature_2m').subtract(273.15).rename('dewpoint_temperature_2m')
    # 4. Zonal Wind (U 10m): m/s
    u10 = img.select('u_component_of_wind_10m').rename('u_wind_10m')
    # 5. Meridional Wind (V 10m): m/s
    v10 = img.select('v_component_of_wind_10m').rename('v_wind_10m')
    # 6. Surface Pressure: Pa -> hPa (/ 100)
    sp = img.select('surface_pressure').divide(100).rename('surface_pressure')
    
    return img.select([]).addBands([tp, t2m, d2m, u10, v10, sp]).copyProperties(img, ["system:time_start"])

def download_era5_land_month(year, month):
    out_dir = folder_induk / str(year)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    nc_path = out_dir / f"era5_land_{year}_{month:02d}.nc"
    if nc_path.exists():
        print(f"ℹ️ File sudah ada: {nc_path.name}, melewati...")
        return
        
    start_date = f"{year}-{month:02d}-01"
    if month == 12:
        end_date = f"{year+1}-01-01"
    else:
        end_date = f"{year}-{month+1:02d}-01"
        
    print(f"\nProcessing Multi-Variable ERA5-Land: {year}-{month:02d} ({start_date} s.d. {end_date})...")
    
    # Filter ERA5-Land Hourly collection
    col = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
           .filterBounds(geometry)
           .filterDate(start_date, end_date)
           .select(["total_precipitation_hourly", "temperature_2m", "dewpoint_temperature_2m", 
                    "u_component_of_wind_10m", "v_component_of_wind_10m", "surface_pressure"]))
    
    count = col.size().getInfo()
    if count == 0:
        print(f"⚠️ Data tidak tersedia untuk {year}-{month:02d}")
        return
        
    # Ekstrak timestamp presisi dari metadata GEE
    timestamps = col.aggregate_array("system:time_start").getInfo()
    dates = pd.to_datetime(timestamps, unit='ms')
    
    # Terapkan pemrosesan konversi satuan & pembentukan band
    col_processed = col.map(process_era5_image)
    
    # Multi-band stacking
    stacked_img = col_processed.toBands()
    
    # Dapatkan URL unduhan GeoTIFF Zip
    url = stacked_img.getDownloadURL({
        'name': f"era5_land_{year}_{month:02d}",
        'region': geometry,
        'scale': 11132,
        'format': 'GEO_TIFF'
    })
    
    # Unduh zip file
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    
    z = zipfile.ZipFile(io.BytesIO(r.content))
    tif_filename = z.namelist()[0]
    temp_tif = f"temp_era5_{year}_{month:02d}.tif"
    z.extract(tif_filename, path=".")
    os.rename(tif_filename, temp_tif)
    
    # Buka TIF dengan rioxarray & pisahkan menjadi multi-variabel NetCDF (time, y, x)
    da = rxr.open_rasterio(temp_tif, masked=True)
    
    var_names = ['precipitation', 'temperature_2m', 'dewpoint_temperature_2m', 'u_wind_10m', 'v_wind_10m', 'surface_pressure']
    num_vars = len(var_names)
    
    # Rekonstruksi xarray Dataset multi-variabel
    ds_out = xr.Dataset()
    for i, var_name in enumerate(var_names):
        # Extract band slice for variable i: i, i+num_vars, i+2*num_vars, ...
        var_da = da.isel(band=slice(i, None, num_vars))
        var_da = var_da.rename({'band': 'time'})
        var_da = var_da.assign_coords(time=dates)
        ds_out[var_name] = var_da
        
    # Simpan sebagai Multi-Variable NetCDF
    ds_out.to_netcdf(nc_path)
    
    da.close()
    ds_out.close()
    if os.path.exists(temp_tif):
        os.remove(temp_tif)
        
    print(f"✅ Selesai ({count} jam x 6 variabel tersimpan): {nc_path.name}")

# Eksekusi Loop
for year in range(tahun_awal, tahun_akhir + 1):
    for month in range(1, 13):
        try:
            download_era5_land_month(year, month)
        except Exception as e:
            print(f"❌ Error {year}-{month:02d}: {e}")